In [1]:
%cd ..


/home/maxim/projects/sandbox/cloud-ivan


In [11]:
import time
import requests
from pathlib import Path

# --- Конфигурация ---
API_BASE_URL = "http://localhost:8000/api/v1/videos"
# Укажи путь к реальному тестовому видео на твоем компьютере
TEST_VIDEO_PATH = Path("data/test/vid/1.mp4") 
RESULT_VIDEO_PATH = Path("data/results/api_test_result.mp4")
FILTER_TYPE = "yolo" # или "pixelate", "grayscale"

def run_e2e_test():
    print(f"[*] Старт E2E теста. Файл: {TEST_VIDEO_PATH.name}")
    
    if not TEST_VIDEO_PATH.exists():
        raise FileNotFoundError(f"Файл не найден: {TEST_VIDEO_PATH}")

    # ==========================================
    # ЭТАП А: Запрос ссылки и прямая загрузка
    # ==========================================
    print("\n[1] Запрашиваем URL для загрузки (Upload URL)...")
    response = requests.post(f"{API_BASE_URL}/upload-url", json={"filename": TEST_VIDEO_PATH.name})
    response.raise_for_status() # Упадет, если статус не 200
    
    data = response.json()
    task_id = data["task_id"]
    upload_url = data["upload_url"].replace("http://minio:9000", "http://localhost:9000")
    print(f"  [+] Task ID получен: {task_id}")
    
    print("\n[2] Выполняем PUT-запрос напрямую в MinIO...")
    with open(TEST_VIDEO_PATH, "rb") as f:
        # Обманываем проверку: шлем трафик на localhost, но подсовываем оригинальный Host
        headers = {"Host": "minio:9000"} 
        put_response = requests.put(upload_url, data=f, headers=headers)
        put_response.raise_for_status()
    print("  [+] Файл успешно загружен в сырой бакет!")

    # ==========================================
    # ЭТАП Б: Запуск задачи
    # ==========================================
    print(f"\n[3] Отправляем команду на обработку (Фильтр: {FILTER_TYPE})...")
    process_response = requests.post(f"{API_BASE_URL}/{task_id}/process", json={"filter_type": FILTER_TYPE})
    process_response.raise_for_status()
    print("  [+] Задача поставлена в очередь Redis!")

    # ==========================================
    # ЭТАП В: Polling (Ожидание) и Скачивание
    # ==========================================
    print("\n[4] Ожидание завершения работы воркера (Polling)...")
    
    download_url = None
    while True:
        status_response = requests.get(f"{API_BASE_URL}/{task_id}")
        status_response.raise_for_status()
        status_data = status_response.json()
        
        current_status = status_data["status"]
        print(f"  ... Статус: {current_status}")
        
        if current_status == "COMPLETED":
            download_url = status_data["download_url"].replace("http://minio:9000", "http://localhost:9000")
            print("  [+] ВИДЕО ГОТОВО!")
            break
        elif current_status == "FAILED":
            raise Exception("Воркер упал с ошибкой! Проверь логи: docker logs yolo_worker")
            
        time.sleep(3) # Ждем 3 секунды перед следующим запросом

    print("\n[5] Скачивание готового результата...")
    RESULT_VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)
    
    # Скачиваем по полученной ссылке
    headers = {"Host": "minio:9000"}
    video_response = requests.get(download_url, stream=True, headers=headers)
    video_response.raise_for_status()
    
    with open(RESULT_VIDEO_PATH, "wb") as f:
        for chunk in video_response.iter_content(chunk_size=8192):
            f.write(chunk)
            
    print(f"  [+] ТЕСТ УСПЕШНО ПРОЙДЕН! Файл сохранен: {RESULT_VIDEO_PATH}")

# Запускаем!
run_e2e_test()


[*] Старт E2E теста. Файл: 1.mp4

[1] Запрашиваем URL для загрузки (Upload URL)...
  [+] Task ID получен: f1dd80ba-0de2-44ef-b9e5-d8d9de2d41d1

[2] Выполняем PUT-запрос напрямую в MinIO...
  [+] Файл успешно загружен в сырой бакет!

[3] Отправляем команду на обработку (Фильтр: yolo)...
  [+] Задача поставлена в очередь Redis!

[4] Ожидание завершения работы воркера (Polling)...
  ... Статус: QUEUED
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: PROCESSING
  ... Статус: COMPLETED
  [+] ВИДЕО ГОТОВО!

[5] Скачивание готового результата...
  [+] ТЕСТ УСПЕШНО ПРОЙДЕН! Файл сохранен: data/results/api_test_result.mp4
